# DPVO on EuRoC (Colab)

Runs Deep Patch Visual Odometry on one EuRoC sequence through the team pipeline:
fetch -> `data/<seq>` layout -> `results/<seq>/dpvo.txt` (TUM) -> shared eval.

**Runtime → Change runtime type → GPU** (a T4 is enough). First session: about
15-20 min (CUDA build + 1.6 GB download). Later sessions reuse the wheel and
the zip cached on Google Drive.

Reference: DPVO's own log reports MH_01_easy ATE (Sim3) of 0.063-0.105 m over
5 trials, median 0.087 m. Landing in that range means the setup works.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/rs-hackatum'  # cache for wheels, data, results
SEQ = 'MH_01_easy'                             # any MH_*, V1_*, V2_* sequence
STRIDE = 2                                     # DPVO's EuRoC setting

In [ ]:
!test -d /content/rs-hackatum-2026 || git clone -q https://github.com/AliAlpOezer/rs-hackatum-2026.git /content/rs-hackatum-2026
!git -C /content/rs-hackatum-2026 pull -q
%cd /content/rs-hackatum-2026

## 1. Install DPVO (cached wheel after the first run)

In [ ]:
!bash tracks/spatial-ai/envs/install_dpvo.sh /content/DPVO {DRIVE}/wheels

## 2. Data: fetch one sequence and convert it to the shared layout

In [ ]:
!pip install -q remotezip
!python data/tools/fetch_euroc.py {SEQ} /content/euroc --cache {DRIVE}/euroc
!python data/tools/euroc_to_seq.py /content/euroc/{SEQ} data/{SEQ}

## 3. Run DPVO

In [ ]:
!python tracks/spatial-ai/runners/dpvo_runner.py data/{SEQ} \
    --network /content/DPVO/dpvo.pth --config /content/DPVO/config/default.yaml \
    --stride {STRIDE} --out results/{SEQ}/dpvo.txt

## 4. Evaluate

In [ ]:
!python eval/evaluate.py data/{SEQ}/gt.txt results/{SEQ}/dpvo.txt

In [ ]:
import copy
import matplotlib.pyplot as plt
from evo.core import sync
from evo.tools import file_interface

ref = file_interface.read_tum_trajectory_file(f'data/{SEQ}/gt.txt')
est = file_interface.read_tum_trajectory_file(f'results/{SEQ}/dpvo.txt')
ref, est = sync.associate_trajectories(ref, est, max_diff=0.01)
est.align(ref, correct_scale=True)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(*ref.positions_xyz[:, :2].T, label='ground truth', color='0.4')
ax.plot(*est.positions_xyz[:, :2].T, label='DPVO (Sim3 aligned)')
ax.set_aspect('equal'); ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.legend(); ax.set_title(SEQ)
fig.savefig(f'results/{SEQ}/dpvo_xy.png', dpi=150, bbox_inches='tight')

## 5. Keep the results

In [ ]:
!mkdir -p {DRIVE}/results && cp -r results/{SEQ} {DRIVE}/results/
!ls -la {DRIVE}/results/{SEQ}